[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lanzlagman/intro-python-astro-PUP/blob/main/notebooks/06_responsible_ai_astrophysics.ipynb)

# Notebook 06: Responsible Use of AI in Astrophysics

**Computational and Data-driven Astrophysics: A Practical Python Workshop**
PUP Physics Society · 15 August 2026

---

A short closing note. 

You're one of the first students who will do an entire degree with a language model available at all times. Nobody is going to tell you not to use these tools; they are genuinely useful and they are not going away. But there are six things worth being deliberate about.

---
## 1. The failure mode is confidence, not error

Language models do not fail by refusing to answer. They fail by producing something **fluent,
plausible and wrong**, which is by far the more dangerous failure, because it looks exactly like
a correct answer.

The example below is an ADQL query of the kind an assistant will happily write for a Gaia cone
search. Before running anything, open the authoritative column list in another tab:

> ### The source of truth
>
> **[Gaia DR3 datamodel: `gaia_source`](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html)**
>
> That page lists every column in the table, with its type, units and definition. It is the
> reference the archive itself is built from, so if a name is not on it, the name does not
> exist. You can also browse the same information interactively at the
> [Gaia archive](https://gea.esac.esa.int/archive/) under *Search → Advanced (ADQL)*, where the
> left-hand panel expands `gaiadr3.gaia_source` into its column list.
>
> **A warning about VizieR.** VizieR serves the Gaia catalogue too, as
> [I/355](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=I/355), and it is a fine place to
> retrieve data. It is *not* the right reference for ADQL column names, because VizieR relabels
> columns to its own house style: `parallax` appears as `Plx`, `pmra` as `pmRA`,
> `phot_g_mean_mag` as `Gmag`. Query the ESA archive with VizieR's names and nothing works.
> Different service, different namespace. Knowing which reference answers which question is
> itself part of the skill.

Now read the query. It looks completely fine.

In [1]:
# A plausible-looking ADQL query of the sort an assistant will happily produce.
suggested_query = """
SELECT source_id, ra, dec, parallax, pmra, pmdec,
       phot_g_mean_mag, bp_rp, teff_val, radial_velocity, distance_pc
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(POINT('ICRS', ra, dec),
                   CIRCLE('ICRS', 56.75, 24.12, 1.0))
"""
print(suggested_query)


SELECT source_id, ra, dec, parallax, pmra, pmdec,
       phot_g_mean_mag, bp_rp, teff_val, radial_velocity, distance_pc
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(POINT('ICRS', ra, dec),
                   CIRCLE('ICRS', 56.75, 24.12, 1.0))



In [2]:
import re

# Fetch the real column list from the archive rather than trusting a list in a notebook.
# This is the same try/except pattern as Notebook 02: live first, cache second.
FALLBACK_DR3_COLUMNS = {
    "source_id", "ra", "dec", "parallax", "parallax_error", "parallax_over_error",
    "pmra", "pmdec", "pmra_error", "pmdec_error", "ruwe",
    "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag", "bp_rp",
    "radial_velocity", "teff_gspphot", "distance_gspphot",
}

try:
    from astroquery.gaia import Gaia
    meta = Gaia.load_table("gaiadr3.gaia_source")
    REAL_DR3_COLUMNS = {c.name for c in meta.columns}
    source = f"live from the ESA archive ({len(REAL_DR3_COLUMNS)} columns)"
except Exception as err:
    REAL_DR3_COLUMNS = FALLBACK_DR3_COLUMNS
    source = f"cached subset, archive unreachable ({type(err).__name__})"

print(f"schema: {source}\n")

selected = re.search(r"SELECT(.*?)FROM", suggested_query, re.S | re.I).group(1)
requested = [c.strip() for c in selected.split(",")]

print("checking each column against the real DR3 schema\n")
bad = []
for col in requested:
    ok = col in REAL_DR3_COLUMNS
    print(f"  {'OK  ' if ok else 'FAIL'}  {col}")
    if not ok:
        bad.append(col)

print(f"\n{len(bad)} column(s) do not exist: {bad}")

schema: live from the ESA archive (152 columns)

checking each column against the real DR3 schema

  OK    source_id
  OK    ra
  OK    dec
  OK    parallax
  OK    pmra
  OK    pmdec
  OK    phot_g_mean_mag
  OK    bp_rp
  FAIL  teff_val
  OK    radial_velocity
  FAIL  distance_pc

2 column(s) do not exist: ['teff_val', 'distance_pc']


This query would have failed against the archive. Worse: had those names happened to exist in
some *other* catalogue, it would have run and silently returned the wrong quantity.

### Check the two failures yourself

| Claimed column | Status | Where to verify |
|---|---|---|
| `teff_val` | Real, but **DR2 only**. Renamed in DR3. | Search the [DR2 datamodel](https://gea.esac.esa.int/archive/documentation/GDR2/Gaia_archive/chap_datamodel/sec_dm_main_tables/ssec_dm_gaia_source.html) for `teff_val`: it is there. Search the [DR3 page](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html): it is gone, replaced by `teff_gspphot`. |
| `distance_pc` | **Never existed**, in any release. | Search either page. Nothing. The nearest real column is `distance_gspphot`, and it is a model-dependent estimate, not a measurement. |

Use your browser's find function on those pages. It takes about fifteen seconds, and that is the
entire point of this section.

> **Why each one is wrong is more interesting than the fact that it is.**
>
> `teff_val` is a **version error**. It was a genuine column, and the model has read a great deal
> of DR2-era code. Effective temperature in DR3 comes from a different processing chain, GSP-Phot,
> so the name changed to `teff_gspphot`. Nothing about the name looks suspicious; you can only
> catch it by checking which release you are querying.
>
> `distance_pc` is a **conceptual error**, and the more revealing of the two. Gaia does not
> measure distance. It measures parallax, and distance is something you *derive* from it, which
> is exactly the subtlety of Notebook 01. The model invented a column that would be convenient
> if the physics worked differently than it does. It is the kind of mistake that sounds like
> expertise.
>
> Both names are *reasonable*, *idiomatic* and *wrong*, and no amount of asking "are you sure?"
> reliably catches either. Asking again just resamples from the same distribution.
>
> **The habit that works:** check the output against a source of truth, whether that is the
> archive schema, the package documentation, or the textbook. Verification is cheap. Here it was
> one `set` and a loop, against a list the archive published itself.

## 2. Cognitive offloading is the real danger

This is the primary risk of using these tools. Unlike obvious code errors, **cognitive offloading**: handing mental tasks to an external tool, silently erodes your core skills. 

LLMs don't just automate tedious chores; they easily take over derivations, debugging, and critical reasoning. Those aren't peripheral tasks; they are the skill itself.

### How Skill Decay Happens
1. **Path of least resistance:** Instant answers always feel cheaper than thinking.
2. **Illusion of competence:** Reading a correct solution creates the sensation of understanding, mimicking mastery right up until you are unaided.
3. **Silent erosion:** Unused skills degrade without you noticing.
4. **Delayed failure:** The loss surfaces late: in exams, interviews, or complex real-world problems.

Learning happens in the friction between a question and an answer. By removing all friction, you risk removing the learning.

> **Use the model for things you *can* do, but don't need to. Do not use it for things you *cannot* do yet.**

* **Offload:** Boilerplate, syntax lookups, decoding tracebacks.
* **Do by hand:** Derivations, problem-set logic, underlying concepts.

**The Test:** *If this tool vanished tomorrow, could I still do this?* If yes, offload it. If no, do it yourself.

## 3. Treat it as a brilliant intern, not an oracle

The best mental model is **an intern who has read everything, experienced nothing, and holds zero accountability.** Vast recall, but no real-world intuition or sense of doubt.

### Rules of Engagement

* **You retain responsibility:** Delegation does not transfer accountability. Your name is on the final work, so you must remain the expert in the room.
* **Be hyper-specific:** Vague prompts yield generic fluff. Detailed context (data size, parameters, exact errors) yields actionable results.
* **Review before running:** Always sanity-check code and logic against units, schemas, or basic physical laws before executing.
* **Demand the reasoning:** Ask for step-by-step logic. It is much easier to catch a flawed process than a flawed final answer.
* **Ignore fake confidence:** The tool never admits doubt or says "I don't know." Asking *"Are you sure?"* does not work; verify independently.
* **Maintain control:** You are the supervisor. Guide the direction of the work; do not let the model dictate what to try next.

## 4. Iteration is fine. Staying in the loop is the condition

Iteratively improving work **you already own** isn't offloading; it’s effective collaboration. 

### Where Iteration Works Best

* **Data Cleaning:** Let the model write repetitive transformations while you supply the target and verify row counts, nulls, and ranges.
* **Verification & Critique:** Invert the dynamic: provide *your* solution and ask the model to spot flaws, bad assumptions, or discrepancies.
* **Articulation:** Refine drafts for clarity. *Rule:* Never publish a claim you couldn't defend if the tool vanished.

### What "In the Loop" Actually Means

A true loop requires your direct input on every single cycle:

| Your Job | Its Job |
|---|---|
| Set goals and constraints | Propose implementations |
| Inspect output against reality | Explain reasoning when asked |
| Direct revisions | Update based on instructions |
| Decide when it is done | *Never delegated* |

> **The Trap:** Slipped handoffs happen silently around cycle four or five when outputs seem consistently fine. Unverified micro-changes quickly accumulate into major drift.

**The Test:** *Can you explain why each change was made?* If yes, you stayed in the loop. If no, go back and re-verify: those unexamined steps are where errors hide.

---
## 5. Reproducibility is the ethics

An impressive result you cannot rerun is not a result.

This applies with or without AI, but AI makes it sharper. If you cannot say where a line of code
came from or why it is correct, you cannot defend it, and someone will ask you to defend it in a
viva or a referee report.

Concretely, and all of it visible in this repository:

| Practice | Where you saw it |
|---|---|
| Pin your versions | `requirements.txt` |
| Cache your data | `data/`, plus every `try/except` in NB02 |
| Derive parameters, do not guess them | the k-distance plot and scored grid in NB05 |
| Save intermediate results | `data/pleiades_query_result.csv` |
| Say what you cut, and why | the quality cuts in NB02 |

That last row is the one people skip. `parallax_over_error > 5` is a **choice**. It changes your
answer. It belongs in your methods section, not buried in a notebook.

---
## 6. Attribution is not optional

**Archives require acknowledgement.** Using Gaia data obliges you to include ESA's
acknowledgement text. This is a condition of use, not a courtesy.

**Software needs citing too.** Astropy, NumPy, SciPy, scikit-learn and matplotlib are all
maintained by people whose careers depend on citation counts. Most publish a preferred citation.

**Open licences have terms.** The teaching resources this workshop draws on, namely Pasha &
Agostino, Zingale and Rougier, are **CC BY-NC-SA 4.0**: credit the original, no commercial use,
and anything built on them carries the same licence. Share-alike propagates. A free student
workshop is squarely within these terms; a paid bootcamp built on the same notebooks would not
be.

**And if a model wrote part of it**, say so. Policies are still settling, but no policy anywhere
has ever punished someone for disclosing too clearly.

---
## Where AI genuinely helps

Not a warning list. These are real, and they are worth using:

- **Boilerplate**: matplotlib styling, argparse scaffolding, docstrings
- **Error messages**: pasting a traceback is often faster than searching
- **Translation**: "what is the pandas equivalent of this IDL or R snippet?"
- **Rubber-ducking**: explaining a bug well enough to ask is half the fix
- **Literature triage**: summarising an abstract to decide whether to read the paper

And where it does not help, and probably will not soon:

- **Deciding what is interesting.** Nothing in this workshop told you the Pleiades was worth
  looking at, or that proper motion was the right pair of axes. That judgement is the job.
- **Knowing when an answer is wrong.** That requires the physics. Which is why you are doing a
  physics degree and not a prompting degree.

---
# That's the workshop

| Notebook | What you can do now |
|---|---|
| **01** | Compute stellar properties with units that protect you |
| **02** | Pull data from any astronomical archive |
| **03** | Integrate a system of ODEs, and know when not to trust it |
| **04** | Build a figure that makes an argument |
| **05** | Find structure nobody labelled, and score it honestly |

### Part 2

A strategizing session, open forum style. Three papers on open clusters, exoplanets and gravitational waves; write the coding plan by hand before any code. Replicate one figure, plan your own project, or both. See `strategizing_session.md`.

### Suggested Readings

1. [Pasha & Agostino, *Python for Astronomers*](https://prappleizer.github.io)
2. [Zingale, *Computational Astrophysics*](https://zingale.github.io/comp_astro_tutorial)
3. [Rougier, *Scientific Visualization*](https://github.com/rougier/scientific-visualization-book)
4. [Ting (2025), *Statistical Machine Learning for Astronomy*](https://arxiv.org/abs/2506.12230)
5. Carroll & Ostlie, throughout.

Full list by notebook in `additional_readings.md`.